In [14]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [17]:
sentences = ['I love Zahra', 'Zahra loves me', 
             'Zahra loves brownies', "Nourou loves training his jawline"]
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

In [18]:
tokenizer.word_index

{'<OOV>': 1,
 'zahra': 2,
 'loves': 3,
 'i': 4,
 'love': 5,
 'me': 6,
 'brownies': 7,
 'nourou': 8,
 'training': 9,
 'his': 10,
 'jawline': 11}

In [19]:
seq = tokenizer.texts_to_sequences(sentences)
seq

[[4, 5, 2], [2, 3, 6], [2, 3, 7], [8, 3, 9, 10, 11]]

In [23]:
padded = pad_sequences(seq, padding='post', truncating='post', maxlen=2)
padded

array([[4, 5],
       [2, 3],
       [2, 3],
       [8, 3]], dtype=int32)

In [67]:
import json

with open("Sarcasm_fixed.json", 'r') as f:
    data = json.load(f)
headlines = []
is_sarcastic = []
for item in data:
    headlines.append(item['headline'])
    is_sarcastic.append(item['is_sarcastic'])

In [ ]:

training_sentences = headlines[:20000]
test_sentences = headlines[20000:]
training_labels = is_sarcastic[:20000]
test_labels = is_sarcastic[20000:]


In [69]:
max_length = 50

In [72]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(training_sentences)
sequences = tokenizer.texts_to_sequences(training_sentences)
padded = pad_sequences(sequences, padding="post", maxlen=max_length) 

test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad= pad_sequences(test_seq, padding="post", maxlen=max_length)

import numpy as np

padded = np.array(padded, dtype=np.float32)
test_pad = np.array(test_pad, dtype=np.float32)
training_labels = np.array(training_labels, dtype=np.float32)
test_labels = np.array(test_labels, dtype=np.float32)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras import layers

model = Sequential([layers.Embedding(output_dim=15, input_dim=len(tokenizer.word_index) + 1),
layers.Bidirectional(layers.LSTM(24)),
layers.Dense(24, activation="relu"),
layers.Dense(1, activation="sigmoid")])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

history = model.fit(padded, training_labels, epochs=5, validation_data=(test_pad, test_labels))

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7930 - loss: 0.4147 - val_accuracy: 0.8554 - val_loss: 0.3364
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9256 - loss: 0.1847 - val_accuracy: 0.8532 - val_loss: 0.3467
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9701 - loss: 0.0845 - val_accuracy: 0.8460 - val_loss: 0.4813
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9866 - loss: 0.0423 - val_accuracy: 0.8454 - val_loss: 0.4962
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9931 - loss: 0.0226 - val_accuracy: 0.8374 - val_loss: 0.7132


In [123]:
def predict(sentence):
    seq = tokenizer.texts_to_sequences([sentence])     
    pad = pad_sequences(seq, maxlen=50, padding='post')
    return model.predict(pad)                  

In [ ]:
predict("")  

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


array([[2.3958624e-05]], dtype=float32)